# Task 1.2 -- Hyperparameter Search

Loads directly from `artifacts/prepared/` so no re-decoding.
Run 24 baseline: ENS val recall=0.811, fpr=0.170, AUC=0.888 (160px, k=16, lr=3e-4, gamma=1.5, AdamW wd=1e-4)

**Note:** each config uses 300s CNN budget for fast comparison. Baseline used 714s.

In [1]:
import os, sys, time
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
sys.path.insert(0, "../solution")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import roc_auc_score

from _lib import seed as _seed
from _lib.calibration import pick_threshold_for_fpr
from _lib.model import build_cnn_bn, cnn_scores, class_weights, batch_to_chw, FocalLoss

PREP = Path("../solution/artifacts/prepared")
IMG_SIZE = 224
TRAIN_IMG_SIZE = 160
CNN_K = 16

def load_split(name):
    n   = int(np.load(PREP / f"n_{name}.npy")[0])
    X   = np.lib.format.open_memmap(str(PREP / f"X_{name}.mmap"), mode="r",
                                     dtype=np.uint8, shape=(n, IMG_SIZE, IMG_SIZE, 3))
    y   = np.load(PREP / f"y_{name}.npy")
    src = np.load(PREP / f"src_{name}.npy")
    F   = np.load(PREP / f"F_{name}.npy")
    return X, y, src, F

print("Loading splits...")
X_fit,  y_fit,  _,  F_fit  = load_split("fit")
X_hold, y_hold, _,  F_hold = load_split("hold")
X_cal,  y_cal,  _,  F_cal  = load_split("cal")
X_val,  y_val,  _,  F_val  = load_split("val")
mean = np.load(PREP / "mean.npy")
std  = np.load(PREP / "std.npy")
print(f"fit={len(X_fit)}  hold={len(X_hold)}  cal={len(X_cal)}  val={len(X_val)}")

Loading splits...
fit=26718  hold=2970  cal=1924  val=1124


In [2]:
def run_experiment(label, budget_s=300, lr=3e-4, weight_decay=1e-4, gamma=1.5,
                   make_scheduler=None, batch=64):
    _seed.set_deterministic(0)
    t0 = time.monotonic()
    cnn = build_cnn_bn(k=CNN_K)
    deadline = t0 + budget_s

    opt = torch.optim.AdamW(cnn.parameters(), lr=lr, weight_decay=weight_decay)
    sched = make_scheduler(opt) if make_scheduler else None
    w = class_weights(y_fit)
    loss_fn = FocalLoss(gamma=gamma, weight=w)
    yt = torch.from_numpy(y_fit).long()
    n = len(X_fit)
    best = {"recall": -1.0, "state": None}
    last_eval = time.monotonic()

    cnn.train()
    step = 0
    while time.monotonic() < deadline:
        perm = np.random.permutation(n)
        for i in range(0, n, batch):
            if time.monotonic() >= deadline:
                break
            ix = perm[i:i+batch]
            xt = batch_to_chw(ix, X_fit, mean, std, target_size=TRAIN_IMG_SIZE)
            loss = loss_fn(cnn(xt), yt[ix])
            opt.zero_grad(); loss.backward(); opt.step()
            if sched: sched.step()
            step += 1

            if time.monotonic() - last_eval >= 30.0:
                scores = cnn_scores(cnn, X_hold, mean, std, target_size=TRAIN_IMG_SIZE)
                thr = pick_threshold_for_fpr(scores[y_hold==0], target_fpr=0.20)
                rec = float(((scores >= thr) & (y_hold==1)).sum() / (y_hold==1).sum())
                if rec > best["recall"] + 1e-4:
                    best = {"recall": rec, "state": {k: v.clone() for k,v in cnn.state_dict().items()}}
                cnn.train()
                last_eval = time.monotonic()

    if best["state"] is None:
        best["state"] = {k: v.clone() for k,v in cnn.state_dict().items()}
    cnn.load_state_dict(best["state"])

    p_cal = cnn_scores(cnn, X_cal, mean, std, target_size=TRAIN_IMG_SIZE)
    p_val = cnn_scores(cnn, X_val, mean, std, target_size=TRAIN_IMG_SIZE)
    thr   = pick_threshold_for_fpr(p_cal[y_cal==0], target_fpr=0.19)
    rec   = float(((p_val >= thr) & (y_val==1)).sum() / (y_val==1).sum())
    fpr   = float(((p_val >= thr) & (y_val==0)).sum() / (y_val==0).sum())
    auc   = float(roc_auc_score(y_val, p_val))
    elapsed = time.monotonic() - t0
    print(f"{label:45s}  val recall={rec:.3f}  fpr={fpr:.3f}  AUC={auc:.4f}  steps={step}  ({elapsed:.0f}s)")
    return {"label": label, "recall": rec, "fpr": fpr, "auc": auc}

results = []
print("Experiment runner ready.")

Experiment runner ready.


## 1. Learning rate sweep

In [3]:
for lr in (1e-3, 5e-4, 3e-4, 1e-4):
    results.append(run_experiment(f"lr={lr}", budget_s=300, lr=lr))

lr=0.001                                       val recall=0.609  fpr=0.213  AUC=0.7559  steps=397  (311s)
lr=0.0005                                      val recall=0.628  fpr=0.202  AUC=0.8051  steps=410  (312s)
lr=0.0003                                      val recall=0.690  fpr=0.218  AUC=0.8056  steps=387  (312s)
lr=0.0001                                      val recall=0.628  fpr=0.197  AUC=0.7767  steps=393  (312s)


## 2. Cosine LR schedule

In [4]:
# T_max = estimated steps in 300s budget (~0.75s/step -> ~400 steps)
T_MAX = 400

for lr in (3e-4, 1e-3):
    results.append(run_experiment(
        f"lr={lr} cosine", budget_s=300, lr=lr,
        make_scheduler=lambda opt, _lr=lr: torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=T_MAX, eta_min=1e-6)
    ))

lr=0.0003 cosine                               val recall=0.667  fpr=0.202  AUC=0.8044  steps=398  (311s)
lr=0.001 cosine                                val recall=0.683  fpr=0.181  AUC=0.8066  steps=407  (311s)


## 3. Focal loss gamma

In [5]:
for g in (1.0, 1.5, 2.0):
    results.append(run_experiment(f"gamma={g}", budget_s=300, gamma=g))

gamma=1.0                                      val recall=0.675  fpr=0.197  AUC=0.8041  steps=410  (312s)
gamma=1.5                                      val recall=0.661  fpr=0.202  AUC=0.7936  steps=409  (311s)
gamma=2.0                                      val recall=0.663  fpr=0.218  AUC=0.7994  steps=414  (312s)


## 4. Weight decay

In [6]:
for wd in (1e-3, 1e-4, 0.0):
    results.append(run_experiment(f"wd={wd}", budget_s=300, weight_decay=wd))

wd=0.001                                       val recall=0.671  fpr=0.229  AUC=0.8082  steps=402  (311s)
wd=0.0001                                      val recall=0.649  fpr=0.207  AUC=0.7999  steps=421  (312s)
wd=0.0                                         val recall=0.639  fpr=0.197  AUC=0.7884  steps=428  (310s)


## Results

In [7]:
print(f"\n{'Config':45s}  {'recall':>7}  {'fpr':>6}  {'AUC':>7}")
print("-" * 75)
print(f"{'run24 baseline (714s budget)':45s}  {0.811:>7.3f}  {0.170:>6.3f}  {0.888:>7.4f}  <- full budget")
print()
for r in sorted(results, key=lambda x: -x["recall"]):
    print(f"{r['label']:45s}  {r['recall']:>7.3f}  {r['fpr']:>6.3f}  {r['auc']:>7.4f}")


Config                                          recall     fpr      AUC
---------------------------------------------------------------------------
run24 baseline (714s budget)                     0.811   0.170   0.8880  <- full budget

lr=0.0003                                        0.690   0.218   0.8056
lr=0.001 cosine                                  0.683   0.181   0.8066
gamma=1.0                                        0.675   0.197   0.8041
wd=0.001                                         0.671   0.229   0.8082
lr=0.0003 cosine                                 0.667   0.202   0.8044
gamma=2.0                                        0.663   0.218   0.7994
gamma=1.5                                        0.661   0.202   0.7936
wd=0.0001                                        0.649   0.207   0.7999
wd=0.0                                           0.639   0.197   0.7884
lr=0.0005                                        0.628   0.202   0.8051
lr=0.0001                                 